# Feature Engineering for Fraud Detection

This notebook focuses on transforming the raw transaction data into meaningful features for fraud detection. We'll create new features, handle categorical variables, and prepare the data for model training.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from geopy.distance import geodesic
import os
import sys

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.2)

# Configure plot size
plt.rcParams['figure.figsize'] = (12, 8)

# Display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Add the project root to the path so we can import from src
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
from src import config

## Load the Data

In [ ]:
# Load training data
train_data = pd.read_csv(config.TRAIN_DATA_PATH)

# Load test data
test_data = pd.read_csv(config.TEST_DATA_PATH)

print(f'Training data shape: {train_data.shape}')
print(f'Test data shape: {test_data.shape}')

In [ ]:
# Display the first few rows of the training data
train_data.head()

## 1. Time-Based Features

Let's extract time-based features from the transaction timestamp. These features can help identify patterns in fraudulent transactions based on when they occur.

In [ ]:
# Convert transaction time to datetime
train_data['trans_date_trans_time'] = pd.to_datetime(train_data['trans_date_trans_time'])

# Extract time-based features
train_data['hour'] = train_data['trans_date_trans_time'].dt.hour
train_data['day'] = train_data['trans_date_trans_time'].dt.day
train_data['weekday'] = train_data['trans_date_trans_time'].dt.dayofweek
train_data['month'] = train_data['trans_date_trans_time'].dt.month
train_data['year'] = train_data['trans_date_trans_time'].dt.year

# Create is_weekend feature
train_data['is_weekend'] = train_data['weekday'].apply(lambda x: 1 if x >= 5 else 0)

# Create time of day categories
train_data['time_of_day'] = train_data['hour'].apply(lambda x: 
                                        'night' if 0 <= x < 6 else
                                        'morning' if 6 <= x < 12 else
                                        'afternoon' if 12 <= x < 18 else
                                        'evening')

# Display the new features
train_data[['trans_date_trans_time', 'hour', 'day', 'weekday', 'month', 'year', 'is_weekend', 'time_of_day']].head()

Let's analyze the relationship between these time-based features and fraud.

In [ ]:
# Analyze fraud by hour of day
hour_fraud = train_data.groupby('hour')['is_fraud'].mean().reset_index()
hour_fraud.columns = ['Hour', 'Fraud Rate']

plt.figure(figsize=(12, 6))
sns.lineplot(x='Hour', y='Fraud Rate', data=hour_fraud, marker='o')
plt.title('Fraud Rate by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Fraud Rate')
plt.grid(True)

# Add percentage labels
for i, rate in enumerate(hour_fraud['Fraud Rate']):
    plt.text(i, rate + 0.0005, f'{rate:.2%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze fraud by day of week
train_data['day_name'] = train_data['trans_date_trans_time'].dt.day_name()

day_fraud = train_data.groupby(['weekday', 'day_name'])['is_fraud'].mean().reset_index()
day_fraud.columns = ['Weekday', 'Day Name', 'Fraud Rate']
day_fraud = day_fraud.sort_values('Weekday')  # Sort by weekday number

plt.figure(figsize=(12, 6))
sns.barplot(x='Day Name', y='Fraud Rate', data=day_fraud)
plt.title('Fraud Rate by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Fraud Rate')

# Add percentage labels
for i, rate in enumerate(day_fraud['Fraud Rate']):
    plt.text(i, rate + 0.0005, f'{rate:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 2. Distance Calculation

The distance between the cardholder and the merchant can be a strong indicator of fraud. Let's calculate this distance using the latitude and longitude coordinates.

In [ ]:
def calculate_distance(row):
    """
    Calculate the distance between the cardholder and merchant in kilometers
    """
    try:
        cardholder_coords = (row['lat'], row['long'])
        merchant_coords = (row['merch_lat'], row['merch_long'])
        return geodesic(cardholder_coords, merchant_coords).kilometers
    except:
        return np.nan

# Calculate distance for a sample of the data (for performance)
sample_data = train_data.sample(n=10000, random_state=42)
sample_data['distance_km'] = sample_data.apply(calculate_distance, axis=1)

# Display the distance feature
sample_data[['lat', 'long', 'merch_lat', 'merch_long', 'distance_km']].head()

In [ ]:
# Analyze distance vs. fraud
plt.figure(figsize=(12, 6))
sns.boxplot(x='is_fraud', y='distance_km', data=sample_data)
plt.title('Distance Between Cardholder and Merchant by Fraud Status')
plt.xlabel('Is Fraud (1 = Yes, 0 = No)')
plt.ylabel('Distance (km)')
plt.ylim(0, 5000)  # Limit y-axis for better visualization
plt.show()

## 3. Age Calculation

Let's calculate the age of the cardholder at the time of the transaction.

In [ ]:
# Convert DOB to datetime
train_data['dob'] = pd.to_datetime(train_data['dob'])

# Calculate age at the time of transaction
train_data['age'] = train_data.apply(lambda row: (row['trans_date_trans_time'].year - row['dob'].year) - 
                                   ((row['trans_date_trans_time'].month, row['trans_date_trans_time'].day) < 
                                    (row['dob'].month, row['dob'].day)), axis=1)

# Display the age feature
train_data[['dob', 'trans_date_trans_time', 'age']].head()

In [ ]:
# Create age groups
bins = [0, 18, 25, 35, 45, 55, 65, 100]
labels = ['<18', '18-25', '26-35', '36-45', '46-55', '56-65', '65+']
train_data['age_group'] = pd.cut(train_data['age'], bins=bins, labels=labels)

# Analyze fraud by age group
age_fraud = train_data.groupby('age_group')['is_fraud'].mean().reset_index()
age_fraud.columns = ['Age Group', 'Fraud Rate']

plt.figure(figsize=(12, 6))
sns.barplot(x='Age Group', y='Fraud Rate', data=age_fraud)
plt.title('Fraud Rate by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Fraud Rate')

# Add percentage labels
for i, rate in enumerate(age_fraud['Fraud Rate']):
    plt.text(i, rate + 0.0005, f'{rate:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Transaction Amount Features

Let's create features related to transaction amounts, such as the transaction amount relative to the average for that category.

In [ ]:
# Calculate average transaction amount by category
category_avg = train_data.groupby('category')['amt'].mean().to_dict()

# Create feature for transaction amount relative to category average
train_data['amt_to_category_avg'] = train_data.apply(
    lambda row: row['amt'] / category_avg.get(row['category'], 1), axis=1)

# Display the new feature
train_data[['category', 'amt', 'amt_to_category_avg']].head()

In [ ]:
# Analyze the relationship between amt_to_category_avg and fraud
plt.figure(figsize=(12, 6))
sns.boxplot(x='is_fraud', y='amt_to_category_avg', data=train_data)
plt.title('Transaction Amount Relative to Category Average by Fraud Status')
plt.xlabel('Is Fraud (1 = Yes, 0 = No)')
plt.ylabel('Amount / Category Average')
plt.ylim(0, 10)  # Limit y-axis for better visualization
plt.show()

## 5. Handling Categorical Features

Let's identify and prepare categorical features for model training.

In [ ]:
# Identify categorical columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns: {categorical_cols}')

# For demonstration, let's look at the category feature
category_counts = train_data['category'].value_counts()
print(f'
Number of unique categories: {len(category_counts)}')
print('
Top 10 categories:')
print(category_counts.head(10))

In [ ]:
# Analyze fraud rate by category
category_fraud = train_data.groupby('category')['is_fraud'].mean().sort_values(ascending=False).reset_index()
category_fraud.columns = ['Category', 'Fraud Rate']

plt.figure(figsize=(12, 8))
sns.barplot(x='Fraud Rate', y='Category', data=category_fraud)
plt.title('Fraud Rate by Transaction Category')
plt.xlabel('Fraud Rate')
plt.ylabel('Category')

# Add percentage labels
for i, rate in enumerate(category_fraud['Fraud Rate']):
    plt.text(rate + 0.001, i, f'{rate:.2%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Feature Selection

Let's select the most relevant features for our fraud detection model.

In [ ]:
# Select features for model
feature_cols = [
    'amt', 'distance_km', 'age', 'hour', 'day', 'weekday', 'month',
    'is_weekend', 'amt_to_category_avg', 'city_pop', 'category', 'time_of_day'
]

# Create the final dataset for model training
final_data = train_data[feature_cols + ['is_fraud']]

# Display the final dataset
final_data.head()

## 7. Save Processed Data

In [ ]:
# Save the processed training data
final_data.to_csv(config.PROCESSED_TRAIN_DATA_PATH, index=False)
print(f'Processed training data saved to {config.PROCESSED_TRAIN_DATA_PATH}')

# Save the category averages for use during prediction
category_avg_df = pd.DataFrame(list(category_avg.items()), columns=['category', 'amt'])
category_avg_df.to_csv(config.PROCESSED_DATA_DIR / 'category_avg.csv', index=False)
print(f'Category averages saved to {os.path.join(config.PROCESSED_DATA_DIR, 'category_avg.csv')}')

## 8. Process Test Data

In [ ]:
# Apply the same preprocessing steps to the test data
# Convert transaction time to datetime
test_data['trans_date_trans_time'] = pd.to_datetime(test_data['trans_date_trans_time'])

# Extract time-based features
test_data['hour'] = test_data['trans_date_trans_time'].dt.hour
test_data['day'] = test_data['trans_date_trans_time'].dt.day
test_data['weekday'] = test_data['trans_date_trans_time'].dt.dayofweek
test_data['month'] = test_data['trans_date_trans_time'].dt.month
test_data['year'] = test_data['trans_date_trans_time'].dt.year
test_data['is_weekend'] = test_data['weekday'].apply(lambda x: 1 if x >= 5 else 0)
test_data['time_of_day'] = test_data['hour'].apply(lambda x: 
                                        'night' if 0 <= x < 6 else
                                        'morning' if 6 <= x < 12 else
                                        'afternoon' if 12 <= x < 18 else
                                        'evening')

# Calculate distance
test_data['distance_km'] = test_data.apply(calculate_distance, axis=1)

# Calculate age
test_data['dob'] = pd.to_datetime(test_data['dob'])
test_data['age'] = test_data.apply(lambda row: (row['trans_date_trans_time'].year - row['dob'].year) - 
                                 ((row['trans_date_trans_time'].month, row['trans_date_trans_time'].day) < 
                                  (row['dob'].month, row['dob'].day)), axis=1)

# Create feature for transaction amount relative to category average
test_data['amt_to_category_avg'] = test_data.apply(
    lambda row: row['amt'] / category_avg.get(row['category'], 1), axis=1)

# Select the same features
final_test_data = test_data[feature_cols + ['is_fraud']]

# Save the processed test data
final_test_data.to_csv(config.PROCESSED_TEST_DATA_PATH, index=False)
print(f'Processed test data saved to {config.PROCESSED_TEST_DATA_PATH}')

## Summary

In this notebook, we performed feature engineering on the transaction data for fraud detection:

1. **Time-Based Features**: Extracted hour, day, weekday, month, year, is_weekend, and time_of_day from the transaction timestamp.

2. **Distance Calculation**: Calculated the distance between the cardholder and merchant locations.

3. **Age Calculation**: Derived the age of the cardholder at the time of the transaction.

4. **Transaction Amount Features**: Created a feature for transaction amount relative to the category average.

5. **Categorical Features**: Identified and analyzed categorical features like category.

6. **Feature Selection**: Selected the most relevant features for the model.

7. **Data Saving**: Saved the processed data for model training.

These engineered features will help improve the performance of our fraud detection model by providing more meaningful information about the transactions.